# Ограниченный анализ защитного подхода на основе adversarial augmentation

Цель эксперимента — проверить, может ли включение состязательных примеров, сформированных разработанной атакой, повысить устойчивость модели Vision Transformer без существенного ухудшения качества классификации на чистых данных.

Сравниваются два сценария:
1. Baseline: дообучение модели на чистых изображениях.
2. Augmented: дообучение модели на объединённой выборке clean + attacked.

Оценка проводится на clean test и attacked test по метрикам Accuracy и weighted F1-score.

## Импорты и настройки:

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import ViTImageProcessor, ViTForImageClassification
from tqdm.auto import tqdm

# НАСТРОЙКИ:
MODEL_PATH = "melanoma_attacks/checkpoint-12670"
CLEAN_ROOT = "balanced_Image"
ATTACKED_ROOT = "isic_1000_attacked"
SEED = 42
TEST_SIZE = 0.2
BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 3
NUM_WORKERS = 0
CLASS_NAMES = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]
LABEL_MAP = {cls: i for i, cls in enumerate(CLASS_NAMES)}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Фиксация random seed:

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Сбор пар clean/attacked:

In [3]:
def build_pairs_dataframe(clean_root, attacked_root, class_names):
    rows = []

    for cls in class_names:
        clean_cls_dir = os.path.join(clean_root, cls)
        attacked_cls_dir = os.path.join(attacked_root, cls)
        clean_files = {
            f for f in os.listdir(clean_cls_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        }
        attacked_files = {
            f for f in os.listdir(attacked_cls_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        }

        common_files = sorted(clean_files & attacked_files)

        for fname in common_files:
            rows.append({
                "class_name": cls,
                "label": LABEL_MAP[cls],
                "image_id": fname,
                "clean_path": os.path.join(clean_cls_dir, fname),
                "attacked_path": os.path.join(attacked_cls_dir, fname),
            })

    df = pd.DataFrame(rows)
    return df

pairs_df = build_pairs_dataframe(CLEAN_ROOT, ATTACKED_ROOT, CLASS_NAMES)
print("Всего пар найдено:", len(pairs_df))
pairs_df.head()

Всего пар найдено: 1000


,class_name,label,image_id,clean_path,attacked_path
0,AK,0,ISIC_0024468.jpg,balanced_Image\AK\ISIC_0024468.jpg,isic_1000_attacked\AK\ISIC_0024468.jpg
1,AK,0,ISIC_0024470.jpg,balanced_Image\AK\ISIC_0024470.jpg,isic_1000_attacked\AK\ISIC_0024470.jpg
2,AK,0,ISIC_0024511.jpg,balanced_Image\AK\ISIC_0024511.jpg,isic_1000_attacked\AK\ISIC_0024511.jpg
3,AK,0,ISIC_0024646.jpg,balanced_Image\AK\ISIC_0024646.jpg,isic_1000_attacked\AK\ISIC_0024646.jpg
4,AK,0,ISIC_0024654.jpg,balanced_Image\AK\ISIC_0024654.jpg,isic_1000_attacked\AK\ISIC_0024654.jpg


## Быстрая проверка баланса и корректности:

In [ ]:
display(pairs_df["class_name"].value_counts().sort_index())

class_name
AK      125
BCC     125
BKL     125
DF      125
MEL     125
NV      125
SCC     125
VASC    125
Name: count, dtype: int64

## Train/test split:

In [ ]:
train_df, test_df = train_test_split(
    pairs_df,
    test_size=TEST_SIZE,
    stratify=pairs_df["class_name"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("\nTrain balance:")
display(train_df["class_name"].value_counts().sort_index())
print("\nTest balance:")
display(test_df["class_name"].value_counts().sort_index())

Train size: 800
Test size: 200

Train balance:


class_name
AK      100
BCC     100
BKL     100
DF      100
MEL     100
NV      100
SCC     100
VASC    100
Name: count, dtype: int64


Test balance:


class_name
AK      25
BCC     25
BKL     25
DF      25
MEL     25
NV      25
SCC     25
VASC    25
Name: count, dtype: int64

## Формирование baseline и augmented train:

In [6]:
baseline_train_df = train_df[["class_name", "label", "image_id", "clean_path"]].copy()
baseline_train_df = baseline_train_df.rename(columns={"clean_path": "image_path"})
baseline_train_df["source"] = "clean"

aug_clean_df = train_df[["class_name", "label", "image_id", "clean_path"]].copy()
aug_clean_df = aug_clean_df.rename(columns={"clean_path": "image_path"})
aug_clean_df["source"] = "clean"

aug_attacked_df = train_df[["class_name", "label", "image_id", "attacked_path"]].copy()
aug_attacked_df = aug_attacked_df.rename(columns={"attacked_path": "image_path"})
aug_attacked_df["source"] = "attacked"

augmented_train_df = pd.concat([aug_clean_df, aug_attacked_df], ignore_index=True)

clean_test_df = test_df[["class_name", "label", "image_id", "clean_path"]].copy()
clean_test_df = clean_test_df.rename(columns={"clean_path": "image_path"})
clean_test_df["source"] = "clean"

attacked_test_df = test_df[["class_name", "label", "image_id", "attacked_path"]].copy()
attacked_test_df = attacked_test_df.rename(columns={"attacked_path": "image_path"})
attacked_test_df["source"] = "attacked"

print("Baseline train:", len(baseline_train_df))
print("Augmented train:", len(augmented_train_df))
print("Clean test:", len(clean_test_df))
print("Attacked test:", len(attacked_test_df))

Baseline train: 800
Augmented train: 1600
Clean test: 200
Attacked test: 200


## Dataset:

In [7]:
processor = ViTImageProcessor.from_pretrained(MODEL_PATH)

class LesionDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        return pixel_values, label

def make_loader(df, processor, batch_size=16, shuffle=False):
    ds = LesionDataset(df, processor)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS
    )

## Функции обучения и оценки:

In [8]:
def load_model():
    model = ViTForImageClassification.from_pretrained(MODEL_PATH)
    model.to(device)
    return model

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    for pixel_values, labels in tqdm(loader, leave=False):
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(len(loader), 1)

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    for pixel_values, labels in tqdm(loader, leave=False):
        pixel_values = pixel_values.to(device)
        outputs = model(pixel_values=pixel_values)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="weighted")
    report = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)
    cm = confusion_matrix(all_labels, all_preds)

    return {
        "accuracy": acc,
        "f1_weighted": f1,
        "report": report,
        "confusion_matrix": cm
    }

def run_experiment(train_df, clean_test_df, attacked_test_df, title):
    print(f"\n===== {title} =====")

    train_loader = make_loader(train_df, processor, batch_size=BATCH_SIZE, shuffle=True)
    clean_test_loader = make_loader(clean_test_df, processor, batch_size=BATCH_SIZE, shuffle=False)
    attacked_test_loader = make_loader(attacked_test_df, processor, batch_size=BATCH_SIZE, shuffle=False)

    model = load_model()
    optimizer = AdamW(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer)
        print(f"Epoch {epoch + 1}/{EPOCHS} | loss = {loss:.4f}")

    clean_metrics = evaluate_model(model, clean_test_loader)
    attacked_metrics = evaluate_model(model, attacked_test_loader)

    print("\n[Clean test]")
    print(f"Accuracy: {clean_metrics['accuracy']:.4f}")
    print(f"F1-score: {clean_metrics['f1_weighted']:.4f}")

    print("\n[Attacked test]")
    print(f"Accuracy: {attacked_metrics['accuracy']:.4f}")
    print(f"F1-score: {attacked_metrics['f1_weighted']:.4f}")

    return model, clean_metrics, attacked_metrics

##  Запуск baseline:

In [9]:
baseline_model, baseline_clean_metrics, baseline_attacked_metrics = run_experiment(
    train_df=baseline_train_df,
    clean_test_df=clean_test_df,
    attacked_test_df=attacked_test_df,
    title="Baseline: fine-tuning on clean train"
)


===== Baseline: fine-tuning on clean train =====


  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1/3 | loss = 0.1215


  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 2/3 | loss = 0.0296


  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 3/3 | loss = 0.0133


  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]


[Clean test]
Accuracy: 0.9750
F1-score: 0.9748

[Attacked test]
Accuracy: 0.1550
F1-score: 0.1360


## Запуск augmented

In [10]:
aug_model, aug_clean_metrics, aug_attacked_metrics = run_experiment(
    train_df=augmented_train_df,
    clean_test_df=clean_test_df,
    attacked_test_df=attacked_test_df,
    title="Augmented: fine-tuning on clean + attacked train"
)


===== Augmented: fine-tuning on clean + attacked train =====


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1/3 | loss = 2.0734


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 2/3 | loss = 0.5833


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 3/3 | loss = 0.2812


  0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/13 [00:00<?, ?it/s]


[Clean test]
Accuracy: 0.9600
F1-score: 0.9600

[Attacked test]
Accuracy: 0.7000
F1-score: 0.6975


## Итоговая таблица:

In [11]:
results_df = pd.DataFrame([
    {
        "Model": "Baseline (clean train)",
        "Clean test Accuracy": baseline_clean_metrics["accuracy"],
        "Clean test F1": baseline_clean_metrics["f1_weighted"],
        "Attacked test Accuracy": baseline_attacked_metrics["accuracy"],
        "Attacked test F1": baseline_attacked_metrics["f1_weighted"],
    },
    {
        "Model": "Augmented (clean + attacked train)",
        "Clean test Accuracy": aug_clean_metrics["accuracy"],
        "Clean test F1": aug_clean_metrics["f1_weighted"],
        "Attacked test Accuracy": aug_attacked_metrics["accuracy"],
        "Attacked test F1": aug_attacked_metrics["f1_weighted"],
    }
])

results_df

,Model,Clean test Accuracy,Clean test F1,Attacked test Accuracy,Attacked test F1
0,Baseline (clean train),0.975,0.974786,0.155,0.136040
1,Augmented (clean + attacked train),0.960,0.959955,0.700,0.697536
